Install new libraries

In [31]:
#!pip install ddgs trafilatura
#!pip install --upgrade pip


Import Dependencies



In [1]:
import os
from openai import OpenAI 
from dotenv import load_dotenv
import json
from pprint import pprint
from  IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura


load_dotenv()

OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()
MODEL="gpt-4.1-nano"
JUDGE_MODEL="gpt-4.1"

Step1: Define the tools

FUNCTIONS

In [2]:
#Search web function

def search_web(query:str):
    """Search the web using DuckDuckGo and return the top 3 results."""
    ddgs= DDGS()
    results=ddgs.text(query, max_results=3)
    print(f"\u2705 Got results")
    return json.dumps(results, indent=2)




In [3]:
#Fetch URL Function

def fetch_url(url:str):
    """ Fetch the content of URL using trafilatura and return the text content."""
    downloaded=trafilatura.fetch_url(url=url)
    if downloaded:
        text=trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 Got Text: {len(text)} chars")
            return text
    print(f"\u274c Failed to fetch URL:")
    return f"Could not extract text from {url}. Try a different source"

STEP2: Describe as LLM tools

In [4]:
tools=[]

In [5]:
search_web_function={
    "name": "search_web",
    "description": "Searches the web using DuckDucGo browser. returns 3 results.",
    "parameters": {
        "type":"object",
        "properties":{
            "query":{
                "type":"string",
                "description":"Searches the websites for information related to the users query"
            }
        },
        "required":["query"]
    }
    
    }

tools.append({"type":"function","function":search_web_function})


fetch_url_function={
    "name": "fetch_url",
    "description": "Fetches the content of a URL.",
    "parameters": {
        "type":"object",
        "properties":{
            "url":{
                "type":"string",
                "description":"The URL to fetch and extract text from "      }
        },
        "required":["url"]
    }
    
    }
tools.append({"type":"function","function":fetch_url_function})


In [ ]:
tools

#Step3: Tool call handler

In [7]:
def handle_tool_call(tool_calls):
    tool_results=[]
    
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args= json.loads(tool_call.function.arguments)

        print(f"\U0001f527 calling function: {function_name} with args: {args}")

        if function_name=="search_web":
        #Send the notification, i.e. call the tool
            result=search_web(args["query"])
            content=f" Search Results: {result}"
        elif function_name == "fetch_url":
            result=fetch_url(args["url"])
            content=f" Fetched URL content: {result}"
        #elif function_name="Insert_function-3":
           # content=Insert_function-3{args["message"]}"

        else:
            content = f"unknown function: {function_name}"

   # print(f"sent notification: {args['message']}")
        tool_call_result={
            "role":"tool",
            "content": content,
            "tool_call_id":tool_call.id
        }
        tool_results.append(tool_call_result)
    return tool_results

Step4: The System Prompt

In [8]:
#Research agent prompt

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.


You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. You MUST gather information from at least 6 distinct sources before delivering your brief. If you have fewer than 6 sources, keep searching. 

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

It is imperative that "Done:" should be at the start of the final response, so that is can easily be parsed and extracted.
You CANNOT and SHOULD NOt include "Done:" in any part of your response except at the BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

#Step5: The Agentic Loop

In [14]:
def run_research_agent(topic:str, max_iterations: int =10) -> str:
    """
    Run the research agent on a topic and return the research brief

    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops

    Returns:
        The research brief as a string

    """
#Initialize conversation message list with System prompt + Research Task
    print(f"\n\U0001F50D Starting research agent for topic: {topic}\n")
    messages=[{"role": "system", "content": RESEARCH_AGENT_PROMPT},
              {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief: {topic}"}]

    #Loop
    iteration=0
        #1. Call the LLM and get response
    while iteration <max_iterations:
        iteration+=1
        print(f"\n\U0001F4DD Iteration {iteration}:\n")
        response=client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools

    )

        message= response.choices[0].message
        messages.append(message)

        #2. Check if LLM called tools
        if message.tool_calls:
          tool_results = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
          messages.extend(tool_results) #.... add info about tool call response to "context", i.e messages change from append to extend for multiple tool calks
    
    #3. Otherwise: No tools were called, read message content
        else:
            content=message.content or "" #no tool calls         #... invoke the LLM one more time to get its updated response

    #Check if Done, then return
    #Otherwise not yet done, append message
    
            if content.startswith("DONE:"):
                research_brief=content[5:]
                print(f"\n\u2705 Research brief completed")
                return research_brief
            else:
                print(f"  \U0001F4DD Agent is thinking:")
                pprint(content)
        
#4. If we are entering the final iteration, force a final answer
        if (iteration == max_iterations-1):
            print(f"\n\u26A0  Safety Limit Reached. Stopping Research in next iteration")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please provide your final research brief now."})

       
#Fallback return
    return "Research brief incomplete. Maximum iterations reached without a finalizing brief."



###EVALS Run 1

In [ ]:
JUDGE_PROMPT=""" 

# Judge Prompt: Insufficient Source Breadth Failure (TRUE/FALSE)

You are scoring a Research Agent's **deliverable research brief** to determine if there was a **source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Source:** Any distinct, identifiable origin of information referenced in the brief — a named article, report, publication, website, database, tool, filing, organization's data/statement, or a named individual's externally-attributed view (e.g., "per Gartner," "a 2023 Pew study found," "in an interview with Reuters, the CEO said"). Formal citation markers, footnotes, or a bibliography are **not** required — informal attribution counts.

**Distinct source count:** The number of unique sources referenced anywhere in the brief, after deduplication. The same source referenced multiple times counts once. Two different pieces from the same outlet (e.g., two separate NYT articles) count as two. Vague, unattributed claims ("reports suggest," "some analysts believe," "it is widely known") do not count as sources.

**Source breadth failure (label TRUE):**

1. **Fewer than 6 distinct sources:** The brief references fewer than 6 unique, identifiable sources by the counting rules above.

**No source breadth failure (label FALSE):** The brief references at least 6 distinct, identifiable sources by the counting rules above.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.


"""

***TOPICS

In [16]:

TOPICS = [
    "The economic impact of AI-driven automation on manufacturing jobs",
    "Recent advances in mRNA vaccine technology beyond COVID-19",
    "The rise of vertical farming and its viability at scale",
    "How central bank digital currencies could reshape monetary policy",
    "The current state of fusion energy research and commercialization timelines",
    "Water scarcity and desalination technology in the Middle East",
    "The effects of remote work on urban commercial real estate",
    "Advances in autonomous vehicle regulation across different countries",
    "The role of gut microbiome research in treating chronic disease",
    "Geopolitical implications of rare earth mineral supply chains",
]

In [ ]:
## run Evals

import contextlib
import io
import pandas as pd

def judge(brief: str) -> str:
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"<research_brief>\n{brief}\n</research_brief>"},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

results = []
for topic in TOPICS:
    with contextlib.redirect_stdout(io.StringIO()): #Supress print ouput from the agent for cleaner evaluation logs
     brief = run_research_agent(topic)
     verdict = judge(brief)
     results.append({"topic": topic, "verdict": verdict, "brief_len": len(brief)})
     print(f"[{verdict}] {topic}")

df = pd.DataFrame(results)
fail_rate = (df["verdict"] == "TRUE").mean()
print(f"\nSource breadth failure rate: {fail_rate:.0%} ({(df['verdict']=='TRUE').sum()}/{len(df)})")
df

Evals Run 2

In [18]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.


You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. Try to gather information from 5-6 different sources
7. When you have enough information, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

It is imperative that "Done:" should be at the start of the final response, so that is can easily be parsed and extracted.
You CANNOT and SHOULD NOt include "Done:" in any part of your response except at the BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

In [22]:
import contextlib
import io
import pandas as pd

def judge(brief: str) -> str:
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"<research_brief>\n{brief}\n</research_brief>"},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

results = []
for topic in TOPICS:
    with contextlib.redirect_stdout(io.StringIO()): #Supress print ouput from the agent for cleaner evaluation logs
     brief = run_research_agent(topic)
     verdict = judge(brief)
     results.append({"topic": topic, "verdict": verdict, "brief_len": len(brief)})
     print(f"[{verdict}] {topic}")
     print(brief)

df = pd.DataFrame(results)
fail_rate = (df["verdict"] == "TRUE").mean()
print(f"\nSource breadth failure rate: {fail_rate:.0%} ({(df['verdict']=='TRUE').sum()}/{len(df)})")
df


Source breadth failure rate: 60% (6/10)


,topic,verdict,brief_len
0,The economic impact of AI-driven automation on...,FALSE,4655
1,Recent advances in mRNA vaccine technology bey...,FALSE,3422
2,The rise of vertical farming and its viability...,TRUE,4083
3,How central bank digital currencies could resh...,FALSE,4639
4,The current state of fusion energy research an...,FALSE,3584
5,Water scarcity and desalination technology in ...,TRUE,5372
6,The effects of remote work on urban commercial...,TRUE,4496
7,Advances in autonomous vehicle regulation acro...,TRUE,4961
8,The role of gut microbiome research in treatin...,TRUE,2987
9,Geopolitical implications of rare earth minera...,TRUE,5418
